In [1]:
import os


In [2]:
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

True

In [3]:
from langchain_mistralai import MistralAIEmbeddings

In [4]:
from langchain_community.document_loaders import PyPDFLoader

C:\Users\Mirza Shayan Baig\AppData\Local\Temp\ipykernel_19736\4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [5]:
print(os.path.exists("../data/Medical_book.pdf"))

True


In [6]:
loader = PyPDFLoader("../data/Medical_book.pdf")
documents = loader.load()

In [7]:
print(documents[0].page_content)

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
text_split = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=150
)

In [10]:
chunks = text_split.split_documents(documents)

In [11]:
len(chunks)

7481

In [12]:
print(type(chunks))
print(len(chunks))

<class 'list'>
7481


In [13]:
from langchain_mistralai import MistralAIEmbeddings

In [14]:
client = MistralAIEmbeddings(
model="mistral-embed",
 api_key=os.getenv("mistral")
)

d:\medical_chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
text = [chunk.page_content for chunk in chunks]

In [16]:
vector = client.embed_documents(text)

In [17]:
len(vector)

7481

In [19]:
vector

[[-0.040863037109375,
  0.0440673828125,
  0.06890869140625,
  -0.0009965896606445312,
  0.020965576171875,
  0.00839996337890625,
  0.024169921875,
  -0.055206298828125,
  -0.0121917724609375,
  0.0018320083618164062,
  -0.03424072265625,
  0.0372314453125,
  -0.037017822265625,
  -0.01776123046875,
  -0.055206298828125,
  0.051361083984375,
  -0.0002073049545288086,
  -0.0050811767578125,
  0.0355224609375,
  0.017120361328125,
  -0.0263214111328125,
  -0.01486968994140625,
  -0.026641845703125,
  0.00033092498779296875,
  -0.039581298828125,
  -0.007488250732421875,
  -0.0511474609375,
  -0.04534912109375,
  -0.005748748779296875,
  -0.0018720626831054688,
  0.0269622802734375,
  -0.019683837890625,
  -0.0117645263671875,
  0.0247039794921875,
  0.0238494873046875,
  -0.01605224609375,
  -0.0198974609375,
  -0.00630950927734375,
  0.01508331298828125,
  -0.0021533966064453125,
  -0.017333984375,
  -0.011444091796875,
  0.0029544830322265625,
  -0.0251312255859375,
  -0.013427734375,

In [18]:
from pinecone import Pinecone


In [ ]:
key=os.getenv("Pinecone_key")


NameError: name 'os' is not defined

In [20]:
pinecone_client = Pinecone(
    api_key=os.getenv("Pinecone_key")
    )

In [21]:
index = pinecone_client.Index("medical-chatbot-data")

In [23]:
records = []

for i in range(len(chunks)):

    chunk = chunks[i]
    single_vector = vector[i]

    metadata = chunk.metadata.copy()
    metadata["text"] = chunk.page_content

    record = {
        "id": "chunk-" + str(i),
        "values": single_vector,
        "metadata": metadata
    }

    records.append(record)

batch_size = 100

for start in range(0, len(records), batch_size):

    batch = records[start : start + batch_size]

    index.upsert(vectors=batch)

print("All vectors uploaded successfully!")

All vectors uploaded successfully!


In [24]:
index.delete(delete_all=True)